# Vave Pipeline - Setup Guide

## Overview
This guide explains how to set up and run the Vave Geospatial Risk Aggregation Pipeline.

## Prerequisites
1. Unity Catalog `main` with appropriate permissions
2. Access to `/Workspace/Repos/vave` Git repository
3. Sample CSV files from `sample_data` folder in Git repo


## Step 2: Upload Sample CSV Files

**IMPORTANT**: Files must be **manually uploaded** from the Git repository.

### File Location in Git Repo
repo_root/ <br>
└── sample_data/  <br>
* ├── vave_events_2026-04-20.csv 
* ├── vave_events_2026-05-01.csv 
* └── vave_events_2026-05-05.csv

### Upload Destination
**Volume Path**: `/Volumes/main/_raw/external_data/vave_raw/`

### Upload Method

**Databricks UI**:
1. Navigate to **Data** → **Volumes**
2. Browse to `main` → `_raw` → `external_data` → `vave_raw`
3. Click **Upload**
4. Select all 3 CSV files from `sample_data/` folder
5. Confirm upload


Expected CSV Format:
event_id,event_timestamp,latitude,longitude,risk_score<br>
EVT001,2026-04-20 08:15:23,51.5074,-0.1278,45.3<br>
EVT002,2026-04-20 09:22:11,51.5155,-0.1426,62.8

## Step 3: Verify File Upload

**Action**: Run this command to verify files are in the volume:

In [0]:
# In Databricks notebook
display(dbutils.fs.ls("/Volumes/main/_raw/external_data/vave_raw/"))

Expected output: 3 CSV files listed

## Step 4: Run Bronze Ingestion

**Notebook**: `Vave_Bronze_dbt`

**Purpose**: Processes CSV files from volume into Bronze Delta table.

**Action**:
1. Open `Vave_Bronze_dbt` notebook
2. Set widget parameter: `fil_pth_str` = Comma-separated file paths

**Example**:<br>
/Volumes/main/_raw/external_data/vave_raw/vave_events_2026-04-20.csv,<br>/Volumes/main/_raw/external_data/vave_raw/vave_events_2026-05-01.csv,<br>/Volumes/main/_raw/external_data/vave_raw/vave_events_2026-05-05.csv

3. Click **Run All**
4. Verify records inserted into `main.bronze.vave_eventsfeed`

**Expected Output**:
=== Bronze Ingestion - dbt Triggered === <br>📊 Target Table: main.bronze.vave_eventsfeed <br>📝 File Type: csv <br>📁 Files to Process: 3 <br>✓ Inserted 190 records <br>✓ Processed 3 files

**Note**: The notebook tracks processed files in `main.control.processed_files` to prevent duplicate processing.

## Step 5: Transform to Silver Layer

**Purpose**: Enrich Bronze data with H3 geospatial indexing.

**Action**: Run this SQL query in a notebook or SQL editor:


In [0]:
%sql
INSERT INTO main.silver.vave_eventsfeed
SELECT
  event_id,
  event_timestamp,
  latitude,
  longitude,
  risk_score,
  h3_latlng_to_cell_string(latitude, longitude, 8) AS h3_ix,
  DATE(event_timestamp) AS event_dt,
  source_file_name,
  ingestion_ts
FROM main.bronze.vave_eventsfeed
WHERE event_id NOT IN (SELECT event_id FROM main.silver.vave_eventsfeed);

What This Does:

* Adds H3 geospatial index at resolution 8 (~0.46 km hexagon edge)
* Extracts date from timestamp for partitioning
* Prevents duplicate records

Verify:

In [0]:
%sql
SELECT COUNT(*) FROM main.silver.vave_eventsfeed;
-- Expected: 190 records

SELECT DISTINCT h3_ix FROM main.silver.vave_eventsfeed;
-- Expected: ~57 unique H3 cells

## Step 6: Refresh Gold Materialized Views

**Purpose**: Aggregate Silver data into analytics-ready Gold tables.

**Action**:

In [0]:
%sql
-- Refresh both gold materialized views
REFRESH MATERIALIZED VIEW main.gold.daily_risk_cell;
REFRESH MATERIALIZED VIEW main.gold.daily_risk_zone;

Verify:

In [0]:
%sql
SELECT * FROM main.gold.daily_risk_cell
ORDER BY event_dt DESC, total_risk_score DESC
LIMIT 10;

Expected Output:

* Daily risk aggregations by H3 cell
* Multiple dates from sample data (April 20 - May 9, 2026)
* 57 unique H3 cells in London area
* Total of 190 events

## Step 7: View Dashboard

**Dashboard**: Vave Geospatial Risk Analysis NM

**Widgets**:
1. **Total Events** counter: 190
2. **Unique H3 Cells** counter: 57
3. **Average Risk Score** counter: 47.32
4. **Risk Trend Over Time**: Line chart showing daily total risk scores
5. **Top 10 Highest Risk Cells**: Bar chart of H3 cells with highest cumulative risk
6. **Top 20 Risk Records**: Table showing detailed records

**Data Sources**:
* `main.gold.daily_risk_cell`
* `main.gold.daily_risk_zone`

**Access**: Navigate to Dashboards in workspace or use published dashboard URL


## Troubleshooting

### Files Not Processing
**Issue**: Files uploaded but not appearing in Bronze table

**Solution**:
1. Verify files in volume:

In [0]:
dbutils.fs.ls("/Volumes/main/_raw/external_data/vave_raw/")

2 .Check widget parameter `fil_pth_str` is set correctly in Bronze notebook
<br>
3. Verify files not already in control table:


# Vave Pipeline - Setup Guide

## H3 Function Not Found
Issue: `h3_latlng_to_cell_string` function error

Solution: Ensure Databricks Runtime 13.3 LTS or higher (H3 functions built-in)

## Gold Views Empty
Issue: Gold materialized views show no data
Solution:
1. Verify Silver table has data




In [0]:
%sql
SELECT COUNT(*) FROM main.silver.vave_eventsfeed;

2. Refresh materialized views manually
3. Check H3 index column populated:

In [0]:
%sql
SELECT h3_ix FROM main.silver.vave_eventsfeed WHERE h3_ix IS NULL;

## Dashboard Shows Old Data
Issue: Dashboard not reflecting recent data

Solution:

1. Refresh materialized views
2. Refresh dashboard page

## Daily Production Workflow

**Batch Schedule** (Recommended: Daily at 2 AM):

1. **File Arrival**: CSV files land in volume throughout previous day
2. **Bronze Ingestion**: Scheduled job runs `Vave_Bronze_dbt` notebook
   * Discovers new files
   * Processes only unprocessed files (idempotent)
   * Updates control table
3. **Silver Transformation**: Scheduled job enriches with H3 indexing
4. **Gold Refresh**: Materialized views refresh
5. **Dashboard Update**: Dashboard displays updated metrics

**Orchestration Options**:
* **Databricks Jobs** (recommended) - Native scheduling
* **dbt Cloud** - Data transformation orchestration
* **Apache Airflow** - Complex workflow management
* **Azure Data Factory** - Azure-native orchestration

**No Auto-Trigger**: Files landing in volume do NOT automatically trigger pipeline. Manual or scheduled execution required.

---

## Migration to Vave Workspace

**Target Structure**: `/Workspace/Repos/vave/notebooks/`

**Essential Files**:
1. `Vave_Config_Params.py` - Shared configuration
2. `Vave_Setup.py` - Infrastructure setup
3. `Vave_Bronze_Ingestion.py` - Main ingestion notebook

**Migration Steps**:
1. Export all 3 notebooks as `.py` source files
2. Commit to Git repository at `/Workspace/Repos/vave/notebooks/`
3. Update `%run` paths to relative paths (e.g., `./Vave_Config_Params`)
4. Run setup notebook in new workspace
5. Test with sample files

**Dashboard Migration**:
* Export as JSON from current workspace
* Import to Vave workspace
* Reconnect datasets to Unity Catalog tables